# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the "Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells" dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata_dict = dataset.metadata.to_json()
print(f"Dataset Title: {metadata_dict['name']}")
print(f"Description: {metadata_dict['description']}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
print("Available Record Sets (by @id):\n")
record_sets = [recset['@id'] for recset in dataset.metadata.to_json().get('recordSet', [])]
if not record_sets:
    # If not in top-level, try searching for all cr:RecordSet objects in the expanded metadata
    from mlcroissant._src.structure.graph import expand_jsonld
    expanded = expand_jsonld(dataset.metadata.to_json())
    record_sets = [x['@id'] for x in expanded if '@type' in x and ('cr:RecordSet' in x['@type'] or 'RecordSet' in x['@type'])]
for idx, rec in enumerate(record_sets):
    print(f"{idx+1}. {rec}")
    
# Display details and fields for each record set
def print_record_set_fields(rs_id):
    print(f'Fields for Record Set {rs_id}:')
    recset = None
    # Try matching the @id
    for obj in dataset.metadata.to_json().get('recordSet', []):
        if obj['@id'] == rs_id:
            recset = obj
            break
    if recset is None:
        # Try expanded jsonld
        for obj in expanded:
            if obj.get('@id', '') == rs_id:
                recset = obj
                break
    
    if recset is not None and 'field' in recset:
        for f in recset['field']:
            field_obj = None
            # Field may be a dict with @id
            if isinstance(f, dict) and '@id' in f:
                field_obj = f
            elif isinstance(f, str):
                # search expanded
                for obj in expanded:
                    if obj.get('@id', '') == f:
                        field_obj = obj
                        break
            if field_obj:
                print(f"  - Field: {field_obj['@id']}\t(label: {field_obj.get('label', field_obj['@id'])})")
    else:
        print('  No fields found.')

if record_sets:
    for rec in record_sets:
        print_record_set_fields(rec)
else:
    print('No record sets found in this dataset.')

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# For this dataset, the main protein record set is at '@id': 'https://api.app.sen.science/frontiers/7154140/7a5e5ae0-9660-4102-b7ab-39e3bf42b088'
main_record_set_id = 'https://api.app.sen.science/frontiers/7154140/7a5e5ae0-9660-4102-b7ab-39e3bf42b088'
record_sets = [main_record_set_id]
dataframes = {}

for record_set in record_sets:
    print(f"Loading records for record set: {record_set}")
    records = list(dataset.records(record_set=record_set))
    df = pd.DataFrame(records)
    dataframes[record_set] = df
    print(f"RecordSet {record_set} columns:\n", df.columns.tolist())
    print(f"First few records for {record_set}:")
    display(df.head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Replace with the actual numeric field @id from the overview section (example: 'coverage_pct')
# Let's suppose there is a column named 'coverage_pct'. You may need to adjust this to match the field @id.
df = dataframes[main_record_set_id]

# Try to find a numeric field
numeric_candidates = [col for col in df.columns if df[col].dtype in ['float64', 'int64'] or pd.api.types.is_numeric_dtype(df[col])]
print("Numeric candidate columns:", numeric_candidates)

numeric_field = numeric_candidates[0] if numeric_candidates else df.columns[0]

# Filter for demonstration
threshold = df[numeric_field].mean() if pd.api.types.is_numeric_dtype(df[numeric_field]) else 10
filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold}:")
display(filtered_df.head())

# Normalize numeric field
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"Normalized {numeric_field} for filtered records:")
display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Try grouping by a categorical field, e.g., 'description' or 'accession'
group_field_candidates = [col for col in df.columns if (df[col].dtype == object or pd.api.types.is_string_dtype(df[col])) and col != numeric_field]
group_field = group_field_candidates[0] if group_field_candidates else None
if group_field:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
    print(f"Grouped mean of {numeric_field} by {group_field}:")
    display(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(7,4))
plt.hist(df[numeric_field].dropna(), bins=30, alpha=0.7, color='royalblue', edgecolor='k')
plt.xlabel(numeric_field)
plt.ylabel('Frequency')
plt.title(f'Distribution of {numeric_field}')
plt.show()

# If group_field found, boxplot
if group_field:
    top_groups = df[group_field].value_counts().head(5).index
    plt.figure(figsize=(10, 5))
    df[df[group_field].isin(top_groups)].boxplot(column=numeric_field, by=group_field, grid=False, showmeans=True)
    plt.title(f'{numeric_field} distribution by {group_field} (Top 5 Categories)')
    plt.suptitle('')
    plt.xlabel(group_field)
    plt.ylabel(numeric_field)
    plt.show()

## 6. Conclusion
In this notebook, we loaded and explored the FAIR2 dataset on mass spectrometry analyses of extracellular vesicles from human mast cells. Using `mlcroissant`, we examined the dataset structure, loaded record sets by their `@id`, discovered and examined numeric and categorical fields, performed basic data filtering and normalization, and visualized key distributions. This workflow shows how Croissant metadata enables reproducible, FAIR-aligned data exploration and processing.